# Agentic RAG with Vector Search

This notebook uses the shared RAG helpers from `lib` and compares lexical search with semantic search behind the same agentic interface.

## Imports

Keep imports together so the rest of the notebook focuses on the RAG flow.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
# from sentence_transformers import SentenceTransformer
from scripts.embedder import Embedder

from lib.agentic_rag import AgenticRAG
from lib.search import (
    LexicalSearchConfig,
    MinsearchLexicalSearchTool,
    SemanticSearchConfig,
    SQLiteSemanticSearchTool,
)
from lib.sources import load_faq_data

2026-06-24 10:24:06.372111130 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


## Environment

Load API credentials and create the OpenAI client used by the agent.

In [2]:
load_dotenv()

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Knowledge Base

Load the course documents once. Both search tools will index this same data.

In [3]:
documents = load_faq_data()

len(documents)

1350

## Encoder

The semantic search tool uses this encoder for both document embeddings and query embeddings.

In [4]:
# encoder = SentenceTransformer("all-MiniLM-L6-v2")
encoder = Embedder("models/Xenova/all-MiniLM-L6-v2")

## Search Tools

The tools own their indexes. Lexical search uses minsearch here, while semantic search uses a SQLite-backed vector index.

In [9]:
lexical_search_tool = MinsearchLexicalSearchTool.from_documents(
    documents=documents,
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    config=LexicalSearchConfig(
        filter_dict={"course": "llm-zoomcamp"},
        boost_dict={"question": 3.0, "section": 0.5},
    ),
)

semantic_search_tool = SQLiteSemanticSearchTool.from_documents(
    documents=documents,
    encoder=encoder,
    text_fields=["question", "answer"],
    keyword_fields=["course"],
    config=SemanticSearchConfig(filter_dict={"course": "llm-zoomcamp"}),
    db_path="semantic_search.db",
    vector_mode="hnsw"
)

  0%|          | 0/27 [00:00<?, ?it/s]

## Instructions

The agent must search before answering and should only use retrieved context.

In [10]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

Always use the search function at least once before answering.
Use it multiple times if you're not able to find a relevant answer in the first attempt.
If you can't answer using the search results, say that you don't know.
""".strip()

## RAGs

Both agents use the same orchestration. Only the search tool changes.

In [11]:
lexical_rag = AgenticRAG(
    llm_client=openai_client,
    search_tool=lexical_search_tool,
    instructions=instructions,
)

semantic_rag = AgenticRAG(
    llm_client=openai_client,
    search_tool=semantic_search_tool,
    instructions=instructions
)

## Try It

Ask the same question through each search strategy and compare the answers.

In [12]:
question = "This set of lectures being taught is up and running, can I still be part of it??"

In [13]:
lexical_answer = lexical_rag.find_and_reply(question)
print(lexical_answer)

The cost to answer this question was: $0.001196.

The agent used 1 tool call to answer it.

Yes — you can still join even if the lectures have already started.

If your goal is to get a certificate, make sure you submit your project while submissions are still being accepted. Homework isn’t mandatory, but it is recommended and can help with your leaderboard rank.

If you want, I can also point you to the main course repo/lesson pages.


In [14]:
semantic_answer = semantic_rag.find_and_reply(question)
print(semantic_answer)

The cost to answer this question was: $0.001085.

The agent used 1 tool call to answer it.

Yes — you can still join and follow along.

If you want a certificate, though, you need to complete and submit the project while the course is still accepting submissions.
